# Dreamer v1 — Colab Training Notebook

DMC walker_walk で Dreamer v1 を学習する。`Run all` で end-to-end 実行できるよう設計。

**事前準備**: 下の `USER CONFIG` セルだけ編集する。

**目標**: walker_walk return ~500 (公式 ~750)。

**所要時間 (T4 GPU 想定)**: smoke 5-10分、short 1-2時間、full 5-8時間。

## 使い方

1. **ランタイムを GPU に変更**: メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU
2. **リポジトリを Drive に置く**: ローカルの `wm-dreamer-repro/` 全体を `MyDrive/wm-dreamer-repro/` にコピー
3. **下の USER CONFIG セルを編集**: RUN_NAME と RUN_MODE を確認
4. **Run all** または上から順にセル実行

結果（checkpoint, 学習曲線）は Drive の `MyDrive/wm-dreamer-results/<RUN_NAME>/` に保存される。

## Step 0: USER CONFIG

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# USER CONFIG — ここだけ編集
# ═════════════════════════════════════════════════════════════════════

# --- リポジトリの取得元 ---
REPO_SOURCE_MODE = "drive"  # "drive" or "github"
REPO_DRIVE_PATH  = "/content/drive/MyDrive/wm-dreamer-repro"
REPO_GITHUB_URL  = ""  # "github" モード時のみ

# --- 結果保存先 (Drive 内. 自動作成) ---
RESULTS_DRIVE_DIR = "/content/drive/MyDrive/wm-dreamer-results"
RUN_NAME = "walker_walk_run1"

# --- 実行モード ---
#   smoke : 5,000 env_step    (~10分, 動作確認のみ)
#   short : 200,000 env_step  (~1-2時間, 立ち上がり確認)
#   full  : 1,000,000 env_step (~5-8時間, 本番)
RUN_MODE = "full"

# 本番前に smoke を走らせるか (推奨: True)
RUN_SMOKE_BEFORE_MAIN = True

# --- wandb (任意) ---
USE_WANDB     = False
WANDB_API_KEY = ""            # https://wandb.ai/authorize
WANDB_PROJECT = "dreamer-repro"

print(f"RUN_NAME = {RUN_NAME}")
print(f"RUN_MODE = {RUN_MODE}")
print(f"REPO_SOURCE_MODE = {REPO_SOURCE_MODE}")

## Step 1: GPU と環境チェック

In [ ]:
import subprocess, sys

print("=== Python ===")
print(sys.version)

print("\n=== Torch / CUDA ===")
try:
    import torch
    print(f"torch       : {torch.__version__}")
    print(f"cuda avail  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"device      : {torch.cuda.get_device_name(0)}")
        print(f"memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("WARNING: GPU が見えていません。ランタイム→ランタイムのタイプを変更→GPU を選んでください。")
except Exception as e:
    print(f"torch import 失敗: {e}")

print("\n=== nvidia-smi ===")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print(f"nvidia-smi 失敗: {e}")

## Step 2: リポジトリを取得して /content/wm-dreamer-repro/ に配置

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_LOCAL = "/content/wm-dreamer-repro"

if REPO_SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    if not Path(REPO_DRIVE_PATH).exists():
        raise FileNotFoundError(
            f"Drive にリポジトリが見つかりません: {REPO_DRIVE_PATH}\n"
            "ローカルの wm-dreamer-repro/ 全体を Drive にコピーしてください。"
        )

    # /content にコピー (Drive 直接 cd だと I/O が遅い)
    if Path(REPO_LOCAL).exists():
        shutil.rmtree(REPO_LOCAL)
    print(f"copying {REPO_DRIVE_PATH} → {REPO_LOCAL} ...")
    shutil.copytree(
        REPO_DRIVE_PATH, REPO_LOCAL,
        ignore=shutil.ignore_patterns(
            ".venv", "__pycache__", ".git", ".pytest_cache", "checkpoints", "*.pyc",
        ),
    )
    print("copy done.")

elif REPO_SOURCE_MODE == "github":
    if not REPO_GITHUB_URL:
        raise ValueError("REPO_GITHUB_URL を設定してください")
    if Path(REPO_LOCAL).exists():
        shutil.rmtree(REPO_LOCAL)
    subprocess.check_call(["git", "clone", REPO_GITHUB_URL, REPO_LOCAL])

else:
    raise ValueError(f"unknown REPO_SOURCE_MODE: {REPO_SOURCE_MODE}")

os.chdir(REPO_LOCAL)
print(f"\ncwd       : {os.getcwd()}")
print(f"contents  : {sorted(os.listdir('.'))}")

## Step 3: 依存関係インストール

Colab の torch (CUDA 対応版) を残し、それ以外を `pyproject.toml` から入れる。

In [ ]:
import subprocess, sys

# torch は Colab のものを使う (CUDA バージョンが GPU と整合済み)
# プロジェクト依存を入れる際 torch は除外したいので、必要なものを直接 pip で
PROJECT_DEPS = [
    "dm-control>=1.0.40",
    "einops>=0.8.2",
    "gymnasium>=1.3.0",
    "imageio>=2.37.3",
    "mujoco>=3.8.0",
    "pyyaml>=6.0.3",
    "tqdm>=4.67.3",
    "wandb>=0.26.1",
    "pytest>=9.0.3",
]

print("=== installing project dependencies ===")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PROJECT_DEPS])

# プロジェクト本体を editable install (コードへの import を可能に)
print("\n=== installing project (--no-deps to keep Colab torch) ===")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_LOCAL]
)
print("\ndone.")

## Step 4: MuJoCo / DMC 動作確認

Colab GPU 上では `MUJOCO_GL=egl` が安定。失敗した場合は `osmesa` にフォールバック。

In [ ]:
import os
import importlib

# 1) egl を試す
os.environ["MUJOCO_GL"] = "egl"

def test_dmc():
    from dm_control import suite
    env = suite.load("walker", "walk")
    env.reset()
    pixels = env.physics.render(camera_id=0, height=64, width=64)
    assert pixels.shape == (64, 64, 3)
    return pixels

try:
    pixels = test_dmc()
    print(f"MUJOCO_GL=egl: OK, render shape={pixels.shape}")
except Exception as e:
    print(f"MUJOCO_GL=egl: 失敗 ({e}), osmesa にフォールバック...")
    os.environ["MUJOCO_GL"] = "osmesa"
    # dm_control を強制 reimport
    import sys
    for m in list(sys.modules):
        if "dm_control" in m or "mujoco" in m:
            del sys.modules[m]
    pixels = test_dmc()
    print(f"MUJOCO_GL=osmesa: OK, render shape={pixels.shape}")

# 画像を表示して目視確認
import matplotlib.pyplot as plt
plt.imshow(pixels)
plt.title(f"DMC walker_walk (MUJOCO_GL={os.environ['MUJOCO_GL']})")
plt.axis("off")
plt.show()

# プロジェクトの env wrapper も動くか
import sys
if "/content/wm-dreamer-repro/src" not in sys.path:
    sys.path.insert(0, "/content/wm-dreamer-repro/src")
from dreamer.env import DMCEnv
env = DMCEnv("walker", "walk", action_repeat=2, seed=0)
obs = env.reset()
print(f"\nDMCEnv wrapper OK: obs.shape={obs.shape}, dtype={obs.dtype}, action_dim={env.action_dim}")
del env

## Step 5: ユニットテスト (任意, 1-2分)

全モジュールが破綻していないか pytest で確認。

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-x", "-q", "tests/"],
    cwd="/content/wm-dreamer-repro",
)
if result.returncode != 0:
    print("\n⚠️  テスト失敗。ログを確認してください。学習を続行する場合は次のセルへ。")
else:
    print("\n✅ 全テスト通過")

## Step 6: Smoke test (5,000 env_step, ~5-10分)

本番学習に進む前に統合ループが GPU 上で破綻なく回ることを確認。

In [ ]:
import subprocess, sys
if RUN_SMOKE_BEFORE_MAIN:
    print("=== Smoke test (5000 env_step) ===\n")
    result = subprocess.run(
        [sys.executable, "scripts/smoke_train_short.py"],
        cwd="/content/wm-dreamer-repro",
    )
    if result.returncode != 0:
        raise RuntimeError("Smoke test failed. 上のログを確認してください。")
    print("\n✅ Smoke test OK. 本番学習に進めます。")
else:
    print("Smoke test をスキップ (RUN_SMOKE_BEFORE_MAIN=False)")

## Step 7: 本番学習

RUN_MODE に応じて total_steps を調整した config を生成し、checkpoint は Drive に直接書き込む。
途中切断しても保存済みの checkpoint は残る。

In [ ]:
import os, shutil, yaml
from pathlib import Path

# === 結果ディレクトリ準備 (Drive) ===
RUN_DIR_DRIVE = Path(RESULTS_DRIVE_DIR) / RUN_NAME
RUN_DIR_DRIVE.mkdir(parents=True, exist_ok=True)
print(f"results dir: {RUN_DIR_DRIVE}")

# === checkpoints/ を Drive にシンボリックリンク ===
# こうすると agent.save() が直接 Drive に書き込まれ、disconnect しても残る
ckpt_local = Path("/content/wm-dreamer-repro/checkpoints")
ckpt_drive = RUN_DIR_DRIVE / "checkpoints"
ckpt_drive.mkdir(exist_ok=True)
if ckpt_local.is_symlink():
    ckpt_local.unlink()
elif ckpt_local.exists():
    shutil.rmtree(ckpt_local)
ckpt_local.symlink_to(ckpt_drive)
print(f"checkpoints/ → {ckpt_drive}")

# === RUN_MODE に応じた config ===
cfg_path = "/content/wm-dreamer-repro/configs/dmc_walker.yaml"
cfg = yaml.safe_load(open(cfg_path))

if RUN_MODE == "smoke":
    cfg["train"]["total_steps"] = 5_000
    cfg["train"]["prefill_episodes"] = 2
    cfg["train"]["pretrain"] = 20
    cfg["replay"]["batch_size"] = 16
    cfg["replay"]["seq_len"] = 20
    cfg["train"]["eval_every"] = 2_000
    cfg["train"]["save_every"] = 5_000
elif RUN_MODE == "short":
    cfg["train"]["total_steps"] = 200_000
    cfg["train"]["eval_every"] = 10_000
    cfg["train"]["save_every"] = 25_000
elif RUN_MODE == "full":
    cfg["train"]["total_steps"] = 1_000_000
else:
    raise ValueError(f"unknown RUN_MODE: {RUN_MODE}")

runtime_cfg_path = str(RUN_DIR_DRIVE / "config_runtime.yaml")
with open(runtime_cfg_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)
print(f"\nruntime config: {runtime_cfg_path}")
print(f"  total_steps     = {cfg['train']['total_steps']:,}")
print(f"  prefill_episodes= {cfg['train']['prefill_episodes']}")
print(f"  pretrain        = {cfg['train']['pretrain']}")
print(f"  batch/seq       = {cfg['replay']['batch_size']}/{cfg['replay']['seq_len']}")

# === wandb 認証 ===
if USE_WANDB:
    if not WANDB_API_KEY:
        print("\nWARNING: USE_WANDB=True だが WANDB_API_KEY が空。wandb なしで継続。")
        USE_WANDB = False
    else:
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
        os.environ["WANDB_PROJECT"] = WANDB_PROJECT

In [ ]:
# === 学習を実行 ===
import os, sys, importlib

os.chdir("/content/wm-dreamer-repro")
if "/content/wm-dreamer-repro/src" not in sys.path:
    sys.path.insert(0, "/content/wm-dreamer-repro/src")

# モジュールリロード (前セルで設定した env vars を反映させるため)
for mod_name in list(sys.modules):
    if mod_name.startswith("dreamer"):
        del sys.modules[mod_name]

import dreamer.train as train_mod

print(f"=== training start (RUN_MODE={RUN_MODE}) ===\n")
import time
_t0 = time.time()
try:
    train_mod.main(
        config_path=runtime_cfg_path,
        use_wandb=USE_WANDB,
        run_name=RUN_NAME,
    )
    print(f"\n=== training done in {(time.time()-_t0)/3600:.2f} hours ===")
except KeyboardInterrupt:
    print("\n=== training interrupted by user ===")
    print("checkpoints/ は Drive に保存済み。Step 8 で評価可能。")

## Step 8: 最終評価 + 動画生成

最新 checkpoint をロードして 10 episode の eval を実行し、動画を Drive に保存。

In [ ]:
import os, sys, yaml
from pathlib import Path
import numpy as np
import torch

os.chdir("/content/wm-dreamer-repro")
if "/content/wm-dreamer-repro/src" not in sys.path:
    sys.path.insert(0, "/content/wm-dreamer-repro/src")

from dreamer.env import DMCEnv
from dreamer.agent import Agent

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = yaml.safe_load(open(str(RUN_DIR_DRIVE / "config_runtime.yaml")))

ckpt_dir = RUN_DIR_DRIVE / "checkpoints"
ckpts = sorted(ckpt_dir.glob("agent_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
if not ckpts:
    print("checkpoint が見つかりません (学習が早期終了した可能性)")
else:
    latest_ckpt = ckpts[-1]
    print(f"loading: {latest_ckpt.name}")

    eval_env = DMCEnv(
        domain=cfg["env"]["domain"],
        task=cfg["env"]["task"],
        action_repeat=cfg["env"]["action_repeat"],
        seed=42,
    )
    agent = Agent(
        action_dim=eval_env.action_dim,
        cnn_depth=cfg["model"].get("cnn_depth", 32),
        rssm_deter=cfg["model"]["rssm_deter"],
        rssm_stoch=cfg["model"]["rssm_stoch"],
        rssm_hidden=cfg["model"]["rssm_hidden"],
        min_std=cfg["model"]["min_std"],
        num_units=cfg["model"]["num_units"],
        num_layers=cfg["model"]["num_layers"],
        device=device,
    )
    agent.load(str(latest_ckpt))

    # 1) スカラー eval
    final_return = agent.evaluate(eval_env, n_episodes=10, max_steps=500)
    print(f"\nFinal eval mean return (10 episodes): {final_return:.2f}")

    with open(RUN_DIR_DRIVE / "final_eval.txt", "w") as f:
        f.write(f"checkpoint: {latest_ckpt.name}\n")
        f.write(f"total_env_steps: {int(latest_ckpt.stem.split('_')[1])}\n")
        f.write(f"final_eval_return: {final_return:.2f}\n")
        f.write(f"n_episodes: 10\n")

    # 2) 動画生成 (1 episode の生 frames を imageio で gif)
    print("\nrendering eval episode for video...")
    import imageio
    agent.reset()
    obs = eval_env.reset()
    frames = [obs]
    ep_return = 0.0
    for _ in range(500):
        action = agent.act(obs, training=False)
        obs, r, done, _ = eval_env.step(action)
        frames.append(obs)
        ep_return += r
        if done:
            break
    video_path = RUN_DIR_DRIVE / "eval_episode.gif"
    imageio.mimsave(str(video_path), frames, fps=30, loop=0)
    print(f"video saved: {video_path}  (return={ep_return:.2f})")

    # 3) Inline 表示
    from IPython.display import Image, display
    display(Image(filename=str(video_path)))

## Step 9: 結果ファイル一覧

In [ ]:
import subprocess
print(f"=== {RUN_DIR_DRIVE} ===\n")
subprocess.run(["ls", "-laR", str(RUN_DIR_DRIVE)])

## トラブルシューティング

### `MUJOCO_GL` でエラー
Step 4 で自動的に `egl → osmesa` へフォールバックする。それでも失敗するときは `glfw` を試す:
```python
os.environ["MUJOCO_GL"] = "glfw"
```

### Out of Memory (OOM)
`config_runtime.yaml` の `replay.batch_size` を下げる。Step 7 のセルを編集:
```python
cfg["replay"]["batch_size"] = 25  # 元 50
```

### セッション切断 (12時間制限など)
- checkpoint は Drive に保存済み (`{RUN_DIR_DRIVE}/checkpoints/`)
- ただし replay buffer は保存していないため、**完全な resume は未対応**
- 切断時点までの checkpoint で Step 8 の eval は可能
- 続きを学習したい場合は新しい RUN_NAME で最初からやり直すか、agent.load() を train.py に組み込む改修が必要

### 学習が進まない (return が上がらない)
- `wm/recon` が下がっているか → 下がっていなければ encoder/decoder を疑う
- `wm/kl` が free_nats=3.0 に張り付いたまま → posterior と prior が動いていない。RSSM の bug
- 5万 env_step 過ぎても return ~10 のまま → actor/critic 学習が進んでいない。Step 6 の smoke で confirm 済みなら、ハイパラ ((kl_scale, learning rate) を疑う

### 進捗のモニタリング
学習中、別タブで以下を実行すると最新 checkpoint の eval を取れる:
```python
# Step 8 のセルを部分的に実行
```
wandb を使うと曲線がライブ表示されるので推奨。